# 課程 12 - 使用代理草稿本減少聊天歷史

本筆記本展示如何使用 Microsoft Agent Framework 管理長對話中的上下文。隨著對話增長，令牌數量會增加——最終超出模型的上下文窗口。我們通過<strong>上下文摘要模式</strong>和用於持久記憶的<strong>代理草稿本</strong>來解決這個問題。

## 你將學習：
1. <strong>為何上下文管理重要</strong>：了解令牌限制和上下文窗口
2. <strong>上下文感知代理</strong>：建立能管理自身對話上下文的代理
3. <strong>上下文摘要模式</strong>：使用工具濃縮對話歷史
4. <strong>代理草稿本</strong>：能在上下文縮減過程中保留的持久記憶

## 先決條件：
- 配置好環境變量的 Azure OpenAI 設置
- 具備先前課程中基礎代理概念的理解


## 設置


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## 為何上下文管理很重要

每個大型語言模型（LLM）都有一個有限的<strong>上下文視窗</strong>——即它可在單次請求中處理的最大標記數量。在多輪對話過程中：

- **標記數量隨每條用戶訊息和助理回覆線性增加。**
- <strong>提示標記佔大部分成本</strong>，因為整段歷史每輪都會重新發送。
- 最終對話會<strong>超出上下文視窗大小</strong>，模型要麼截斷內容，要麼報錯。

### 管理上下文的策略

| 策略 | 運作方式 | 權衡 |
|---|---|---|
| <strong>截斷</strong> | 丟棄最舊訊息 | 失去早期上下文 |
| <strong>摘要</strong> | 將較早訊息濃縮成摘要 | 會失去部分細節，但保留重點 |
| **速記本 / 外部記憶體** | 將關鍵資訊存放在對話之外 | 需呼叫工具，但可保存任何程度的訊息縮減 |

在本筆記本中，我們結合了<strong>摘要</strong>與<strong>速記本工具</strong>，讓代理即使在對話歷史被濃縮時，亦能維持連貫性。


## 建立一個具上下文感知的代理


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## 模擬長時間對話

我哋一齊行過一個多輪對話，睇吓點樣慢慢累積上下文。代理人應該喺多個回合中保存重要細節（喜好、預算、旅遊日期），並展示連貫性。


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

注意代理如何保留先前對話的上下文 — 它知道關於日本、壽司、寺廟、攝影、3000美元預算、獨自旅行，以及四月份的時間範圍。在短時間的對話中這運作良好，但隨著對話擴展，完整的歷史紀錄重新傳送會變得昂貴。

讓我們繼續更多回合的對話，以觀察上下文的累積：


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 上下文摘要模式

隨著對話的進展，我們可以使用 <strong>摘要工具</strong> 將累積的上下文濃縮成簡潔的格式。代理人調用此工具來記錄關鍵偏好，即使較舊的訊息被刪除，重要資訊仍然得以保留。

此模式是更複雜歷史縮減的基礎：
1. 代理人從對話中識別出關鍵事實
2. 調用摘要工具將其持久化
3. 可以安全移除較舊的訊息，因為摘要已捕捉重要內容

以下我們定義了 `summarize_preferences` 工具，代理人可調用此工具來記錄已學習內容的簡明摘要。


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 總結

在本課程中，您學會了如何使用 Microsoft Agent Framework 管理長時間運行代理對話中的上下文：

### 主要概念
- <strong>上下文視窗是有限的</strong> — 對話歷史中的每個標記都會產生成本並計入限制。
- <strong>摘要工具</strong> 讓代理能將累積的上下文濃縮成精簡摘要，減少標記使用同時保留重要資訊。
- <strong>代理草稿板</strong> 提供持久的外部記憶，能在任何對話縮減後仍保留內容。

### 您建立的內容
- 一個 <strong>上下文感知代理</strong>，能在多輪對話中維持連續性
- 一個 <strong>摘要工具</strong> (`summarize_preferences`)，用緊湊格式記錄用戶關鍵資訊
- 一個展示上下文保留及變更處理的 <strong>多輪對話</strong>

### 實際應用
- <strong>客服機械人</strong>：在長時間支援會話中記住偏好設定
- <strong>個人助理</strong>：追蹤進行中的專案，無需重複說明上下文
- <strong>教育導師</strong>：在多次互動中維持學生進度

### 下一步
- 實作具備檔案持久化的完整草稿板工具
- 在摘要後新增自動歷史截斷功能
- 與向量資料庫結合以實現語意記憶搜尋
- 建立能在數日後以完整上下文恢復對話的代理


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責聲明**：
本文件使用 AI 翻譯服務 [Co-op Translator](https://github.com/Azure/co-op-translator) 進行翻譯。雖然我們力求準確，但請注意，自動翻譯可能包含錯誤或不準確之處。原始文件的母語版本應被視為權威來源。對於重要資訊，建議尋求專業人工翻譯。我們不對因使用本翻譯而引起的任何誤解或曲解承擔責任。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
